We keep the DummyTransformerBlock and DummyLayerNorm blank, and pass the input to the model and receives the logits as output

![image_1769518945790.png](./image_1769518945790.png "image_1769518945790.png")

![image_1769518868260.png](./image_1769518868260.png "image_1769518868260.png")

![image_1769519177913.png](./image_1769519177913.png "image_1769519177913.png")

![image_1769518888596.png](./image_1769518888596.png "image_1769518888596.png")

![image_1769518920577.png](./image_1769518920577.png "image_1769518920577.png")

In [0]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [0]:
%pip install torch

In [0]:
%pip install tiktoken

In [0]:
import torch
import torch.nn as nn

NOTE: The forward method in your DummyGPTModel class will be called when you pass input data to an instance of the model using the call method, which is inherited from nn.Module. For example, if you have an instance model and a tensor in_idx, you would call model(in_idx). This automatically invokes the forward method and returns the output logits. This is standard PyTorch behavior

In [0]:
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        print(cfg)
        #{'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': False}
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"]) #50257, 768
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])#1024,768 
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        # Use a placeholder for TransformerBlock
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        # Use a placeholder for LayerNorm
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        print(batch_size, seq_len) #2,4
        tok_embeds = self.tok_emb(in_idx)
        print("token embedding",tok_embeds)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        print("position embedding",pos_embeds)
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [0]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # A simple placeholder

    def forward(self, x):
        # This block does nothing and just returns its input.
        return x

In [0]:
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # The parameters here are just to mimic the LayerNorm interface.

    def forward(self, x):
        # This layer does nothing and just returns its input.
        return x

In [0]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

In [0]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [0]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)